# Feature Pipeline con Feast _(version detallada)_

_Definir, registrar y servir features con un feature store open-source._

**Modulo 1 — Feature Engineering & Feature Stores** | DSRP Machine Learning Engineering
**Profesor:** Miguel Arquez

![Feature Pipeline Feast](assets/header.png)

> 📖 **Version detallada** — Incluye comentarios linea a linea en cada bloque de codigo
> y ampliaciones teoricas en las secciones de contexto.  
> Para una lectura mas rapida, consulta la version original `03_feature_pipeline_feast.ipynb`.

## Introduccion

Hasta ahora calculamos features de forma ad-hoc: cada notebook las recrea desde cero.
Esto genera dos problemas en produccion:

1. **Sesgo entrenamiento-serving:** el codigo que calcula features en entrenamiento
   suele diferir del que las calcula en produccion (distintos equipos, distintas
   versiones). El modelo recibe entradas distintas a las que aprendio y se degrada.
2. **Falta de reusabilidad:** cada equipo reimplementa las mismas features.

Un **feature store** resuelve ambos problemas:
- Define las features **una sola vez** en codigo.
- Las sirve de forma identica tanto para entrenamiento (offline) como para inferencia
  online.

**Feast** (Feature Store) es la solucion open-source mas adoptada. Su arquitectura:

```
+-------------+         +------------------+         +--------------+
| Datos crudos|  -----> | OFFLINE store    |  -----> | ONLINE store |
| (CSV/parq.) |   FE    | (parquet/Redshift|  mat.   | (Redis/DDB)  |
+-------------+         +------------------+         +--------------+
                                 |                          |
                    get_historical_features        get_online_features
                         (entrenamiento)               (inferencia)
```

### Conceptos clave de Feast

| Concepto | Descripcion |
|---|---|
| **Entity** | La clave de negocio (p.ej. `house_id`). Une features con etiquetas. |
| **FileSource** | Fuente de datos offline (parquet con `event_timestamp`). |
| **Field** | Una feature individual (nombre + tipo de dato). |
| **FeatureView** | Grupo de features con TTL, Entity y Source. La unidad de servicio. |
| **Registry** | Base de datos interna de Feast con todas las definiciones. |
| **feast apply** | Sincroniza el codigo con el registry ("despliega" las definiciones). |
| **feast materialize** | Lleva datos del offline al online store para serving de baja latencia. |
| **Point-in-time** | Al construir training sets, solo usa features disponibles antes del evento. |

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

# Dibujamos la arquitectura de Feast con matplotlib para que sea ejecutable
# sin dependencias externas. El diagrama muestra el flujo de datos:
# crudos -> offline store -> online store; y los dos caminos de acceso.
fig, ax = plt.subplots(figsize=(13, 5.5))
ax.set_xlim(0, 13); ax.set_ylim(0, 6); ax.axis("off")

def rect(x, y, w, h, label, color, fs=9):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.08",
                                fc=color, ec="#555", lw=1.4))
    ax.text(x+w/2, y+h/2, label, ha="center", va="center", fontsize=fs, wrap=True)

def arr(p1, p2, lbl="", col="#333"):
    ax.add_patch(FancyArrowPatch(p1, p2, arrowstyle="-|>",
                                 mutation_scale=16, color=col, lw=1.5))
    if lbl:
        mx, my = (p1[0]+p2[0])/2, (p1[1]+p2[1])/2
        ax.text(mx, my+0.22, lbl, ha="center", fontsize=8, color=col, style="italic")

# Bloques de la arquitectura
rect(0.1,  3.5, 2.0, 1.5, "Datos crudos\n(CSV / parquet)",     "#cfe8ff", 9)
rect(3.0,  3.5, 3.0, 1.5, "OFFLINE store\n(parquet + timestamps)", "#d6f5d6", 9)
rect(8.0,  3.5, 4.5, 1.5, "ONLINE store\n(Redis — ultimo valor)",  "#ffe2b3", 9)
rect(3.0,  1.5, 3.0, 1.2, "REGISTRY\n(feature_store.yaml)",    "#f3d1ff", 9)
rect(0.1,  0.1, 5.5, 1.0, "get_historical_features\n(entrenamiento, point-in-time)", "#e8e8e8", 8)
rect(7.0,  0.1, 5.5, 1.0, "get_online_features\n(inferencia, ms latencia)",         "#e8e8e8", 8)

# Flechas
arr((2.1, 4.25), (3.0, 4.25), "FE +\nparquet", "#1f77b4")
arr((6.0, 4.25), (8.0, 4.25), "feast\nmaterialize", "#54a24b")
arr((4.5, 3.5), (4.5, 2.7))
arr((4.5, 1.5), (2.8, 1.1), "mismas\nfeatures", "#1f77b4")
arr((4.5, 1.5), (9.0, 1.1), "mismas\nfeatures", "#d62728")

ax.text(6.5, 5.6, "Arquitectura de Feast: offline + online store",
        ha="center", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## Setup y datos

Cargamos Ames Housing y calculamos un conjunto de features numericas y categoricas
que registraremos en Feast. El helper `load_housing()` busca el CSV y, si no existe,
genera datos sinteticos para que el notebook corra sin conexion.

In [ ]:
import numpy as np
import pandas as pd
import os
from datetime import datetime


def load_housing():
    """Localiza housing_train.csv en rutas candidatas o genera datos sinteticos."""
    candidates = []
    if os.environ.get("HOUSING_CSV"):
        candidates.append(os.environ["HOUSING_CSV"])
    candidates += [
        os.path.join("..", "..", "data", "housing_train.csv"),
        os.path.join("..", "..", "..", "data", "housing_train.csv"),
        os.path.join("data", "housing_train.csv"),
    ]
    for path in candidates:
        if path and os.path.exists(path):
            df = pd.read_csv(path)
            print(f"Dataset cargado: {path}  shape={df.shape}")
            return df
    print("CSV no encontrado — datos sinteticos.")
    rng = np.random.default_rng(0)
    n = 400
    oq = rng.integers(3, 10, n)
    gla = (oq * 180 + rng.normal(400, 250, n)).clip(400, 5500)
    sale = (oq * 22000 + gla * 55 + rng.normal(0, 25000, n)).clip(40000, 600000)
    return pd.DataFrame({
        "Id": np.arange(1, n+1),
        "OverallQual": oq,
        "GrLivArea": gla.round().astype(int),
        "TotalBsmtSF": (gla*0.6+rng.normal(0,150,n)).clip(0,3000).round().astype(int),
        "1stFlrSF": (gla*0.55+rng.normal(0,100,n)).clip(400,2000).round().astype(int),
        "GarageCars": rng.integers(0, 4, n),
        "GarageArea": (rng.integers(0,4,n)*200+rng.normal(0,50,n)).clip(0,1500).round().astype(int),
        "YearBuilt": rng.integers(1900, 2010, n),
        "LotArea": rng.gamma(2.0, 5000, n).clip(1300, 60000).round().astype(int),
        "FullBath": rng.integers(1, 4, n),
        "Neighborhood": rng.choice(["NAmes","CollgCr","OldTown","Edwards","Gilbert"], n),
        "ExterQual": rng.choice(["TA","Gd","Ex","Fa"], n),
        "SalePrice": sale.round().astype(int),
    })


df = load_housing()
print(f"Columnas: {list(df.columns[:10])} ...")

## Paso 1 — Feature Engineering y guardado en parquet

Aplicamos transformaciones a los datos crudos para producir el **DataFrame de features**
que guardamos como parquet. Este parquet es el **offline store** de Feast: la fuente
de verdad historica con timestamps.

### ¿Por que parquet y no CSV?

- Parquet conserva los tipos de dato (fechas, enteros, floats) sin conversion.
- Es mucho mas eficiente en disco (columnar, comprimido).
- Feast requiere que el campo `event_timestamp` sea de tipo `datetime64[us, UTC]`
  (timestamp con timezone UTC).

### ¿Que es `event_timestamp`?

Es el campo que Feast usa para la **correctitud point-in-time**: al construir el set
de entrenamiento, para cada entidad solo se usan features con `event_timestamp` anterior
o igual al timestamp de la etiqueta. Esto simula que en el momento de la venta solo
conocias las features calculadas antes de esa fecha.

In [ ]:
# REPO: ruta al directorio del feature repo de Feast.
# Es donde se encuentran features.py y feature_store.yaml.
REPO = os.path.abspath(os.path.join("..", "platform", "feature_repo"))

# PARQUET_PATH: destino del parquet del offline store.
# Feast lo usa como FileSource en features.py.
PARQUET_PATH = os.path.join(REPO, "data", "housing_features.parquet")

print(f"REPO       : {REPO}")
print(f"PARQUET    : {PARQUET_PATH}")

# Construimos el DataFrame de features desde el DataFrame original de Ames.
feat = pd.DataFrame()

# house_id: la ENTIDAD de Feast. Identifica de forma unica cada casa.
# Feast usa este campo para hacer los joins al servir features.
feat["house_id"] = df["Id"].astype(int)

# event_timestamp: cuando fueron validas estas features (punto en el tiempo).
# Feast usa este campo para la correctitud point-in-time.
# Usamos UTC + timezone-aware para compatibilidad con el standard de Feast.
feat["event_timestamp"] = pd.to_datetime("today", utc=True)

# --- Features numericas (sin transformacion especial) ---
# overall_qual: puntuacion de calidad general (1-10). Alta correlacion con SalePrice.
feat["overall_qual"] = df["OverallQual"].fillna(5).astype(int)

# gr_liv_area: superficie habitable en sqft. Feature mas predictiva de SalePrice.
feat["gr_liv_area"] = df["GrLivArea"].fillna(df["GrLivArea"].median()).astype(int)

# total_bsmt_sf: superficie total del sotano. Puede ser 0 si no hay sotano.
feat["total_bsmt_sf"] = df["TotalBsmtSF"].fillna(0).astype(int)

# first_flr_sf: superficie del primer piso (sqft).
col_1st = "1stFlrSF" if "1stFlrSF" in df.columns else "first_flr_sf"
feat["first_flr_sf"] = df[col_1st].fillna(0).astype(int) if col_1st in df.columns else 0

# garage_cars: capacidad del garaje (numero de coches). NaN = sin garaje -> 0.
feat["garage_cars"] = df["GarageCars"].fillna(0).astype(int)

# garage_area: superficie del garaje (sqft). NaN = sin garaje -> 0.
feat["garage_area"] = df["GarageArea"].fillna(0).astype(int)

# year_built: ano de construccion. Features temporales importantes en inmuebles.
feat["year_built"] = df["YearBuilt"].fillna(1960).astype(int)

# lot_area: superficie del lote en sqft. Variable con alta varianza y sesgo.
feat["lot_area"] = df["LotArea"].fillna(df["LotArea"].median()).astype(int)

# full_bath: numero de banos completos.
feat["full_bath"] = df["FullBath"].fillna(1).astype(int)

# --- Feature categorica ---
# neighborhood: barrio de la propiedad. Alta cardinalidad (25 barrios).
feat["neighborhood"] = df["Neighborhood"].fillna("Other").astype(str)

# --- Feature ordinal ---
# exter_qual_ord: calidad exterior codificada como ordinal (None=0, Po=1, ..., Ex=5).
# La codificacion ordinal preserva el orden significativo de la escala de calidad.
QUALITY_MAP = {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5}
feat["exter_qual_ord"] = (
    df["ExterQual"].fillna("None").map(QUALITY_MAP).fillna(0).astype(int)
)

# --- Variable objetivo ---
# sale_price: el precio de venta (no es una feature que Feast sirva online,
# pero lo incluimos en el parquet para armar el set de entrenamiento facilmente).
feat["sale_price"] = df["SalePrice"].fillna(df["SalePrice"].median()).astype(int)

print(f"DataFrame de features: {feat.shape}")
print(f"Columnas: {list(feat.columns)}")
feat.head(3)

In [ ]:
# Guardamos las features en parquet (el OFFLINE store de Feast).
# Feast leeera este parquet cuando hagamos get_historical_features.

# Creamos el directorio si no existe (makedirs con exist_ok=True es seguro).
os.makedirs(os.path.dirname(PARQUET_PATH), exist_ok=True)

# index=False: no guardamos el indice de pandas (no es una columna de datos).
feat.to_parquet(PARQUET_PATH, index=False)

# Verificamos el guardado leyendo el parquet y comprobando el numero de filas.
verify = pd.read_parquet(PARQUET_PATH)
print(f"Parquet guardado: {PARQUET_PATH}")
print(f"Verificacion: {verify.shape} filas x columnas")
print(f"event_timestamp dtype: {verify['event_timestamp'].dtype}")

## Paso 2 — Desplegar definiciones con `feast apply`

`feast apply` lee el archivo `features.py` en el repo y sincroniza las definiciones
con el **registry** (la base de datos interna de Feast). Es el equivalente a un
`git push` de las definiciones de features.

El archivo `features.py` define:
- La `Entity` (`house_id`)
- El `FileSource` (ruta al parquet + campo de timestamp)
- El `FeatureView` (grupo de features + TTL + entity + source)

```python
# Extracto de features.py
house = Entity(name="house_id", value_type=ValueType.INT64)

source = FileSource(path="data/housing_features.parquet",
                    timestamp_field="event_timestamp")

house_fv = FeatureView(
    name="house_features",
    entities=[house],
    ttl=timedelta(days=3650),
    schema=[Field(name="gr_liv_area", dtype=Int64), ...],
    source=source,
)
```

**TTL (Time To Live):** cuanto tiempo hacia atras puede buscar Feast al servir una
feature. Con TTL=10 anos, el online store sirve el ultimo valor disponible sin importar
cuando fue calculado.

In [ ]:
import subprocess

# Ejecutamos 'feast apply' en el directorio del feature repo.
# subprocess.run: ejecuta un comando externo desde Python.
# capture_output=True: captura stdout y stderr para mostrarlos.
# text=True: decodifica la salida como texto (en lugar de bytes).
result = subprocess.run(
    ["feast", "apply"],      # comando: 'feast apply'
    cwd=REPO,                # directorio de trabajo: el feature repo
    capture_output=True,
    text=True,
)

# Mostramos la salida de feast apply.
# Si todo esta bien, veremos 'Registered entity house_id' y 'Registered feature view house_features'.
if result.stdout:
    print("--- feast apply stdout ---")
    print(result.stdout)
if result.stderr:
    print("--- feast apply stderr ---")
    print(result.stderr)

# result.returncode: 0 = exito, != 0 = error.
print(f"Return code: {result.returncode}")
if result.returncode != 0:
    print("AVISO: feast apply fallo. Comprueba que feast este instalado")
    print("y que el repo exista en:", REPO)

## Paso 3 — Construir el training set con `get_historical_features`

`get_historical_features` realiza un **join point-in-time** entre el `entity_df`
(que dice 'que entidades y en que momento') y el offline store.

### Mecanismo interno

Para cada fila del `entity_df` con (`house_id=X`, `event_timestamp=T`), Feast busca
en el offline store la feature mas reciente de la casa X cuyo `event_timestamp <= T`.
Esto garantiza que no se filtre informacion del futuro (*no leakage temporal*).

```
entity_df:                    offline store:
+----------+----------+        +----------+----------+--------+
| house_id | event_ts |  JOIN  | house_id | event_ts | feat   |
+----------+----------+  -->   +----------+----------+--------+
| 1        | T1       |        | 1        | T0<T1    | val_A  |
| 2        | T2       |        | 2        | T0<T2    | val_B  |
+----------+----------+        +----------+----------+--------+
```

In [ ]:
# Lista de features a recuperar. Formato: 'nombre_del_feature_view:nombre_de_feature'.
# Estos nombres deben coincidir exactamente con los definidos en features.py.
FEATURES = [
    "house_features:overall_qual",
    "house_features:gr_liv_area",
    "house_features:total_bsmt_sf",
    "house_features:first_flr_sf",
    "house_features:garage_cars",
    "house_features:garage_area",
    "house_features:year_built",
    "house_features:lot_area",
    "house_features:full_bath",
    "house_features:neighborhood",
    "house_features:exter_qual_ord",
]

try:
    from feast import FeatureStore

    # Instanciamos el FeatureStore: lee feature_store.yaml y el registry.
    store = FeatureStore(repo_path=REPO)

    # entity_df: le decimos a Feast QUE casas queremos y EN QUE MOMENTO.
    # Usamos un subconjunto de casas para la demo.
    sample_ids = feat["house_id"].values[:10]

    # event_timestamp=utcnow(): queremos las features validas "ahora".
    # En un escenario de entrenamiento real, este seria el timestamp de la venta.
    entity_df = pd.DataFrame({
        "house_id": sample_ids,
        "event_timestamp": pd.to_datetime([datetime.utcnow()] * len(sample_ids), utc=True),
    })

    # get_historical_features: join point-in-time entre entity_df y el offline store.
    # .to_df(): convierte el resultado a un DataFrame de pandas.
    training_df = store.get_historical_features(
        entity_df=entity_df,
        features=FEATURES,
    ).to_df()

    print(f"Training set (via Feast): {training_df.shape}")
    print(training_df.head(3))

except Exception as e:
    # Si Feast no esta instalado o el registry no esta aplicado,
    # leemos el parquet directamente (mismos datos).
    print(f"Feast no disponible ({type(e).__name__}: {e})")
    print("Usando el parquet directamente (misma fuente de datos que Feast)...")
    training_df = pd.read_parquet(PARQUET_PATH).head(10)
    print(f"Training set (via parquet): {training_df.shape}")
    print(training_df.head(3))

## Paso 4 — Materializar al online store con `feast materialize-incremental`

El **online store** (Redis o SQLite en modo local) guarda el **ultimo valor** de
cada feature por entidad. Esto permite servir features con latencia de milisegundos
en produccion.

`feast materialize-incremental` lleva datos del **offline al online store**,
actualizando solo los registros nuevos desde la ultima materializacion (incremental).

```
offline store (historia completa)  --materialize-->  online store (solo el ultimo valor)
```

**¿Por que separar offline y online?**
- El offline store (parquet/Redshift) optimiza para *scans historicos* de millones de filas.
- El online store (Redis) optimiza para *lookups por clave* en < 5ms.
- No puedes usar el offline store en produccion: es demasiado lento para inferencia en tiempo real.

In [ ]:
# Materializamos: llevamos datos del offline store al online store.
# materialize-incremental: solo materializa los datos NUEVOS desde la ultima vez.
# end_date: hasta que timestamp materializar (en produccion, tipicamente "ahora").
end_date = datetime.utcnow().isoformat()

result_mat = subprocess.run(
    ["feast", "materialize-incremental", end_date],
    cwd=REPO,
    capture_output=True,
    text=True,
)

if result_mat.stdout:
    print("--- feast materialize-incremental stdout ---")
    print(result_mat.stdout)
if result_mat.stderr:
    print("--- feast materialize-incremental stderr ---")
    print(result_mat.stderr)

print(f"Return code: {result_mat.returncode}")
if result_mat.returncode != 0:
    print("AVISO: materialize fallo. Requiere feast apply previo y Redis/SQLite online.")

## Paso 5 — Recuperar features online con `get_online_features`

`get_online_features` sirve el **ultimo valor materializado** para cada entidad desde
el online store. Es la interfaz de serving: cuando un modelo necesita features para
hacer una prediccion, llama a este metodo con el `house_id` y recibe las features
en < 10ms.

### Diferencias clave con `get_historical_features`

| | `get_historical_features` | `get_online_features` |
|---|---|---|
| Fuente | Offline store (parquet/Redshift) | Online store (Redis) |
| Latencia | Segundos a minutos | < 10ms |
| Devuelve | Historia completa (point-in-time) | Solo el ULTIMO valor |
| Uso | Construir training sets | Inferencia en tiempo real |
| Requiere materialize | No | Si |

In [ ]:
# ID de la casa para la que queremos features online (simula una peticion de serving).
EXAMPLE_ID = int(feat["house_id"].iloc[0])

try:
    from feast import FeatureStore
    store = FeatureStore(repo_path=REPO)

    # get_online_features: recupera el ULTIMO valor de cada feature
    # para la lista de entidades especificada.
    # entity_rows: lista de diccionarios {entity_name: entity_value}.
    online_response = store.get_online_features(
        features=FEATURES,
        entity_rows=[{"house_id": EXAMPLE_ID}],
    )

    # .to_dict(): convierte la respuesta a un diccionario Python.
    online_dict = online_response.to_dict()

    print(f"Features online para house_id={EXAMPLE_ID}:")
    for k, v in online_dict.items():
        print(f"  {k}: {v}")

except Exception as e:
    # Fallback: si el online store no esta disponible, mostramos la fila del parquet.
    print(f"Online store no disponible ({type(e).__name__}: {e})")
    print("Mostrando la fila del parquet para house_id={EXAMPLE_ID}:")
    fila = feat[feat["house_id"] == EXAMPLE_ID].iloc[0]
    for col in FEATURES:
        # Los feature refs tienen formato 'view:feature'; extraemos solo el nombre de la feature.
        feat_name = col.split(":")[1]
        if feat_name in fila.index:
            print(f"  {feat_name}: {fila[feat_name]}")

## Repaso — el ciclo de vida de una feature en Feast

```
1. DEFINICION  -> features.py (Entity + FileSource + FeatureView)
2. DEPLOYMENT  -> feast apply (sincroniza con el registry)
3. INGESTION   -> feat.to_parquet() (datos al offline store)
4. HISTORICAL  -> get_historical_features (training set point-in-time)
5. MATERIALIZE -> feast materialize-incremental (offline -> online)
6. ONLINE      -> get_online_features (serving en ms)
```

**La garantia clave:** los pasos 4 y 6 usan las *mismas* definiciones de features
(mismo FeatureView, mismo codigo de FE). Por tanto, el modelo ve en produccion
exactamente la misma representacion con la que aprendio.

### Conexion con los otros notebooks

- **Notebook 1** calculamos las transformaciones (encoding, escalado, imputacion);
  aqui las empaquetamos en el parquet del offline store.
- **Notebook 2** detectamos outliers; la version limpiada de `GrLivArea` entra al
  DataFrame de features.
- **Notebook 4** usamos `get_historical_features` (o el parquet directamente) para
  armar el training set y entrenamos el modelo de regresion.